# Calgary 311 Service Requests: SQL Analysis
## Part 3: Fixing department names and comparing communities fairly

**Section A** fixes the department-name problem from Part 2. The City renamed departments during 2021–2025, so the same service appears under different names. We map every service to its **current** department, so trends compare like with like.

**Section B** answers the question most 311 analyses skip: *which communities really have high demand?* Raw counts mostly reflect how many people live somewhere. We join 2021 census population data and compare **requests per 1,000 residents**.

**New SQL skills in this part:** joining two different datasets, `LEFT JOIN` to measure match rates, `ROW_NUMBER()` with `PARTITION BY` for "top N per group", comparing two rankings, and `CORR()` for correlation.

## Setup: connect to the database

In [ ]:
!pip -q install duckdb

import os, datetime
import requests
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/calgary_311_project"
except Exception as e:
    print("Drive not available, using local folder:", e)
    PROJECT_DIR = "calgary_311_project"

DB_PATH = os.path.join(PROJECT_DIR, "calgary_311.duckdb")
SQL_DIR = os.path.join(PROJECT_DIR, "sql"); RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(SQL_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

con = duckdb.connect(DB_PATH)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

def q(sql):
    return con.sql(sql).df()

def run(name, sql):
    with open(os.path.join(SQL_DIR, f"{name}.sql"), "w") as f:
        f.write(sql.strip() + "\n")
    df = con.sql(sql).df()
    df.to_csv(os.path.join(RESULTS_DIR, f"{name}.csv"), index=False)
    return df

# Same rules as Part 2: drop "TO BE DELETED", flag duplicates
con.sql("""
    CREATE OR REPLACE VIEW requests_v AS
    SELECT *, status IN ('Duplicate (Closed)', 'Duplicate (Open)') AS is_duplicate
    FROM requests
    WHERE status <> 'TO BE DELETED'
""")
print(f"{q('SELECT COUNT(*) AS n FROM requests_v')['n'][0]:,} requests ready")

---
# Section A: Fixing the department names

### A1. Confirm the problem
This counts how many service types were handled under **more than one** department name during the five years.

In [ ]:
run("07a_services_with_multiple_agencies", """
    WITH service_agencies AS (
        SELECT service_name,
               COUNT(DISTINCT agency) AS agency_names,
               COUNT(*)               AS requests
        FROM requests_v
        WHERE NOT is_duplicate
        GROUP BY service_name
    )
    SELECT
        COUNT(*)                                          AS total_service_types,
        COUNT(*) FILTER (WHERE agency_names > 1)          AS types_with_multiple_agency_names,
        ROUND(100.0 * SUM(requests) FILTER (WHERE agency_names > 1) / SUM(requests), 1)
                                                          AS pct_of_requests_affected
    FROM service_agencies
""")

An example: the department name used each year for the largest affected services.

In [ ]:
q("""
    WITH multi AS (
        SELECT service_name
        FROM requests_v
        WHERE NOT is_duplicate
        GROUP BY service_name
        HAVING COUNT(DISTINCT agency) > 1
        ORDER BY COUNT(*) DESC
        LIMIT 5
    )
    SELECT r.service_name, r.request_year, r.agency, COUNT(*) AS requests
    FROM requests_v r
    JOIN multi m ON m.service_name = r.service_name
    WHERE NOT r.is_duplicate
    GROUP BY r.service_name, r.request_year, r.agency
    ORDER BY r.service_name, r.request_year, requests DESC
""")

### A2. Map each service to its current department
For each service, we find the department name used **most recently**. `ROW_NUMBER() OVER (PARTITION BY service_name ORDER BY last_seen DESC)` numbers the department names *within each service*, newest first, and we keep number 1. This "latest record per group" pattern is one of the most common SQL interview questions.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE service_current_agency AS
    WITH agency_last_seen AS (
        SELECT service_name, agency, MAX(requested_at) AS last_seen
        FROM requests_v
        GROUP BY service_name, agency
    ),
    ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY service_name ORDER BY last_seen DESC) AS rn
        FROM agency_last_seen
    )
    SELECT service_name, agency AS current_agency
    FROM ranked
    WHERE rn = 1
""")

con.sql("""
    CREATE OR REPLACE VIEW requests_mapped AS
    SELECT r.*, m.current_agency
    FROM requests_v r
    JOIN service_current_agency m ON m.service_name = r.service_name
""")
q("""
    SELECT COUNT(DISTINCT agency) AS original_agency_names,
           COUNT(DISTINCT current_agency) AS current_agency_names
    FROM requests_mapped
""")

### A3. Redo the department trend with consistent names
This is the Part 2 query 3c again, now grouped by the current department. More departments should have data for all five years, so the comparison is fairer.

In [ ]:
trend = run("07b_agency_trend_mapped", """
    WITH yearly AS (
        SELECT current_agency, request_year,
               COUNT(*)                                                      AS closed_requests,
               MEDIAN(resolution_days)                                       AS median_days,
               PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY resolution_days)  AS p90_days
        FROM requests_mapped
        WHERE NOT is_duplicate AND resolution_days IS NOT NULL
        GROUP BY current_agency, request_year
        HAVING COUNT(*) >= 1000
    )
    SELECT
        current_agency,
        MAX(p90_days) FILTER (WHERE request_year = 2021) AS p90_2021,
        MAX(p90_days) FILTER (WHERE request_year = 2023) AS p90_2023,
        MAX(p90_days) FILTER (WHERE request_year = 2025) AS p90_2025,
        MAX(p90_days) FILTER (WHERE request_year = 2025)
          - MAX(p90_days) FILTER (WHERE request_year = 2021) AS change_2021_to_2025
    FROM yearly
    GROUP BY current_agency
    HAVING COUNT(*) = 5
    ORDER BY change_2021_to_2025 DESC
""")
trend

---
# Section B: Requests per 1,000 residents

### B1. Download the 2021 census population by community
This is the City's *2021 Federal Census Population and Dwellings by Community* dataset. It's small, so this takes seconds.

In [ ]:
CENSUS_URL = "https://data.calgary.ca/resource/f9wk-wej9.csv"
CENSUS_PATH = os.path.join(PROJECT_DIR, "census_2021_by_community.csv")

if not os.path.exists(CENSUS_PATH):
    r = requests.get(CENSUS_URL, params={"$limit": 100000}, timeout=300)
    r.raise_for_status()
    with open(CENSUS_PATH, "wb") as f:
        f.write(r.content)

con.sql(f"CREATE OR REPLACE TABLE census_raw AS SELECT * FROM read_csv('{CENSUS_PATH}', header = true)")
print(f"{q('SELECT COUNT(*) AS n FROM census_raw')['n'][0]:,} rows downloaded")
q("DESCRIBE census_raw")

In [ ]:
q("SELECT * FROM census_raw LIMIT 5")

### B2. Identify the columns and validate
The cell below guesses which columns hold the **community code**, **community name**, and **total population**. Then it runs a **validation check**: the populations should add up to about 1.31 million, Calgary's 2021 census population.

If the check fails, **send me the output of the two cells above** and I'll tell you what to put in the three override settings below.

In [ ]:
# Leave these as None to auto-detect. Set them to column names if auto-detection fails.
CODE_COL_OVERRIDE = None
NAME_COL_OVERRIDE = None
POP_COL_OVERRIDE  = None

CITY_POP_2021 = 1_306_780
cols = q("DESCRIBE census_raw")
names = list(cols["column_name"])
numeric = [c for c, t in zip(cols["column_name"], cols["column_type"])
           if any(k in t.upper() for k in ("INT", "DOUBLE", "DECIMAL", "FLOAT", "REAL"))]

def find(pred, pool):
    return next((c for c in pool if pred(c.lower())), None)

CODE_COL = CODE_COL_OVERRIDE or find(lambda c: "comm" in c and "code" in c, names) or find(lambda c: c in ("code", "community_code"), names)
NAME_COL = NAME_COL_OVERRIDE or find(lambda c: ("comm" in c or "community" in c) and "name" in c, names) or find(lambda c: c in ("name", "community", "comm_name"), names)

def col_total(c):
    return float(q(f'SELECT SUM("{c}") AS s FROM census_raw')["s"][0] or 0)

if POP_COL_OVERRIDE:
    POP_COL = POP_COL_OVERRIDE
else:
    POP_COL = find(lambda c: "pop" in c and "total" in c, numeric)
    if POP_COL is None:   # otherwise pick the numeric column whose total is closest to the city population
        totals = {c: col_total(c) for c in numeric}
        close = {c: abs(t - CITY_POP_2021) / CITY_POP_2021 for c, t in totals.items() if t}
        POP_COL = min(close, key=close.get) if close else None

print("Community code column:", CODE_COL)
print("Community name column:", NAME_COL)
print("Population column:    ", POP_COL)

rows, distinct = q(f'SELECT COUNT(*) AS n, COUNT(DISTINCT "{CODE_COL or NAME_COL}") AS d FROM census_raw').iloc[0]
total_pop = col_total(POP_COL) if POP_COL else 0
print(f"\nRows: {rows:,} | distinct communities: {distinct:,} | population total: {total_pop:,.0f}")

CHECK_PASSED = (POP_COL is not None and (CODE_COL or NAME_COL) is not None
                and rows == distinct and abs(total_pop - CITY_POP_2021) / CITY_POP_2021 < 0.10)
if CHECK_PASSED:
    print("Validation PASSED: one row per community, and the total matches Calgary's 2021 population.")
else:
    print("Validation FAILED. Please send the outputs of this cell and the two cells above, and don't continue yet.")

### B3. Build a clean population table
Community codes are standardised to upper case with no spaces, matching how we cleaned the 311 data in Part 1.

In [ ]:
assert CHECK_PASSED, "Fix the validation in B2 first."
key = CODE_COL or NAME_COL
con.sql(f"""
    CREATE OR REPLACE TABLE community_population AS
    SELECT
        UPPER(TRIM(CAST("{key}" AS VARCHAR)))                             AS join_key,
        {f'UPPER(TRIM(CAST("{NAME_COL}" AS VARCHAR)))' if NAME_COL else 'NULL'} AS census_name,
        CAST("{POP_COL}" AS BIGINT)                                        AS population
    FROM census_raw
""")
JOIN_ON = "comm_code" if CODE_COL else "community"
print("Joining 311 requests to the census on:", JOIN_ON)
q("SELECT * FROM community_population ORDER BY population DESC LIMIT 5")

### B4. How well do the two datasets match?
A `LEFT JOIN` keeps every 311 request even when no census row matches. Counting the unmatched ones tells us how much data the per-capita analysis covers. Communities created after 2022 won't be in the census, which is expected.

In [ ]:
run("08a_census_match_rate", f"""
    SELECT
        COUNT(*)                                             AS requests_with_community,
        COUNT(p.join_key)                                    AS matched_to_census,
        ROUND(100.0 * COUNT(p.join_key) / COUNT(*), 1)       AS pct_matched
    FROM requests_v r
    LEFT JOIN community_population p ON p.join_key = r.{JOIN_ON}
    WHERE NOT r.is_duplicate AND r.{JOIN_ON} IS NOT NULL
""")

In [ ]:
# The unmatched communities with the most requests (likely new or renamed communities)
q(f"""
    SELECT r.community, r.comm_code, COUNT(*) AS requests
    FROM requests_v r
    LEFT JOIN community_population p ON p.join_key = r.{JOIN_ON}
    WHERE NOT r.is_duplicate AND r.{JOIN_ON} IS NOT NULL AND p.join_key IS NULL
    GROUP BY r.community, r.comm_code
    ORDER BY requests DESC
    LIMIT 10
""")

### B5. Requests per 1,000 residents
**Design choices, explained in the README:**
- We use requests from **2021 and 2022**, averaged per year, because they're closest to the 2021 census. Fast-growing new suburbs have gained residents since then, so using later years would overstate their rates.
- We only include communities with **at least 1,000 residents**. In areas with very few residents, such as the downtown core or industrial parks, most requests come from workers and visitors, so a per-resident rate would be misleading.

The query ranks communities two ways, by raw count and by per-capita rate, so we can see how much the picture changes.

In [ ]:
MIN_POP = 1000
percap = run("08b_requests_per_1000_residents", f"""
    WITH community_requests AS (
        SELECT {JOIN_ON} AS join_key, MAX(community) AS community,
               COUNT(*) / 2.0 AS avg_requests_per_year
        FROM requests_v
        WHERE NOT is_duplicate AND request_year IN (2021, 2022) AND {JOIN_ON} IS NOT NULL
        GROUP BY {JOIN_ON}
    ),
    joined AS (
        SELECT c.community, p.population, c.avg_requests_per_year,
               1000.0 * c.avg_requests_per_year / p.population AS per_1000
        FROM community_requests c
        JOIN community_population p ON p.join_key = c.join_key
        WHERE p.population >= {MIN_POP}
    )
    SELECT
        community,
        population,
        ROUND(avg_requests_per_year, 0)                          AS avg_requests_per_year,
        ROUND(per_1000, 1)                                       AS requests_per_1000_residents,
        ROUND(per_1000 / AVG(per_1000) OVER (), 2)               AS times_city_average,
        RANK() OVER (ORDER BY per_1000 DESC)                     AS per_capita_rank,
        RANK() OVER (ORDER BY avg_requests_per_year DESC)        AS raw_count_rank
    FROM joined
    ORDER BY per_capita_rank
""")
print(f"{len(percap)} communities with {MIN_POP:,}+ residents")
percap.head(15)

In [ ]:
top = percap.head(15).sort_values("requests_per_1000_residents")
ax = top.plot(x="community", y="requests_per_1000_residents", kind="barh", legend=False, figsize=(8, 5), color="#dd8452")
ax.axvline(percap["requests_per_1000_residents"].mean(), color="grey", ls="--", lw=1)
ax.set_title("Highest 311 demand per 1,000 residents (2021–22 avg; dashed = community average)")
ax.set_xlabel("Requests per 1,000 residents per year"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

And the lowest-demand communities:

In [ ]:
percap.tail(10)

### B6. How different are the two rankings?
If per-capita analysis just repeated the raw ranking, it wouldn't add much. This counts how many of the top 15 by raw count are also in the top 15 per capita.

In [ ]:
run("08c_ranking_overlap", """
    SELECT
        COUNT(*) FILTER (WHERE raw_count_rank <= 15)                          AS top15_by_raw_count,
        COUNT(*) FILTER (WHERE raw_count_rank <= 15 AND per_capita_rank <= 15) AS also_top15_per_capita
    FROM read_csv_auto('{path}')
""".format(path=os.path.join(RESULTS_DIR, "08b_requests_per_1000_residents.csv")))

### B7. What drives demand in the highest per-capita communities?
"Top N per group" again: `ROW_NUMBER() OVER (PARTITION BY community ORDER BY requests DESC)` ranks service types *within* each community, and we keep the top 3 for each of the five highest per-capita communities.

In [ ]:
top5 = "', '".join(percap.head(5)["community"].str.replace("'", "''"))
run("08d_top_services_in_high_demand_communities", f"""
    WITH counts AS (
        SELECT community, service_name, COUNT(*) AS requests
        FROM requests_v
        WHERE NOT is_duplicate AND request_year IN (2021, 2022)
          AND community IN ('{top5}')
        GROUP BY community, service_name
    ),
    ranked AS (
        SELECT *,
               ROUND(100.0 * requests / SUM(requests) OVER (PARTITION BY community), 1) AS pct_of_community,
               ROW_NUMBER() OVER (PARTITION BY community ORDER BY requests DESC)        AS rn
        FROM counts
    )
    SELECT community, rn AS rank_in_community, service_name, requests, pct_of_community
    FROM ranked
    WHERE rn <= 3
    ORDER BY community, rn
""")

### B8. Is service speed fair across communities?
The median and 90th-percentile resolution time for every community with at least 2,000 closed requests. Then `CORR()` checks whether communities with **higher demand per resident** also wait **longer**. A correlation near 0 means no relationship, and close to +1 means high-demand areas wait longer.

In [ ]:
speed = run("08e_resolution_time_by_community", """
    SELECT
        community,
        COUNT(*)                                                      AS closed_requests,
        MEDIAN(resolution_days)                                       AS median_days,
        PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY resolution_days)  AS p90_days
    FROM requests_v
    WHERE NOT is_duplicate AND resolution_days IS NOT NULL AND community IS NOT NULL
    GROUP BY community
    HAVING COUNT(*) >= 2000
    ORDER BY p90_days DESC
""")
print("Slowest communities (90th percentile):"); display(speed.head(10))
print("Fastest communities:"); display(speed.tail(5))

In [ ]:
con.register("percap_df", percap)
con.register("speed_df", speed)
run("08f_demand_vs_speed_correlation", """
    SELECT
        COUNT(*)                                                         AS communities,
        ROUND(CORR(p.requests_per_1000_residents, s.median_days), 2)     AS corr_demand_vs_median_days,
        ROUND(CORR(p.requests_per_1000_residents, s.p90_days), 2)        AS corr_demand_vs_p90_days
    FROM percap_df p
    JOIN speed_df s ON s.community = p.community
""")

---
## Checkpoint

**Send me the outputs of:** A1 (both tables), A3, B2 (the validation message), B4 (both tables), B5 (top 15 and bottom 10), B6, B7, B8, and the correlation.

After this, Part 4 packages everything: a GitHub repository with your SQL files, a README of findings and recommendations, and the resume entry.

In [ ]:
con.close()
print("Done. All queries and results are saved in", PROJECT_DIR)